# atomic-povray Prototype 1

The geometry stage is deliberately separate. Run it once, then reuse `geometry` while experimenting with style and camera settings. `DisplayBounds.fractional_ranges` combines replication, offset, and fractional cropping: each `(min, max)` pair is expressed along one unit-cell vector. Default bond rules are generated during geometry construction, including asymmetric metal→non-metal boundary extension.

In [ ]:
from pathlib import Path
from IPython.display import Image
from atomic_povray import *

project = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
poscar = project / 'tests' / 'data' / 'fe2o3-012-1x1-relaxed.vasp'
povray = r'C:\Program Files\POV-Ray\v3.7\bin\pvengine64.exe'

In [ ]:
structure = load_structure(poscar)
geometry = build_geometry(
    structure,
    bounds=DisplayBounds(
        fractional_ranges=((-2.0, 2.0), (-1.5, 1.5), (0.45, 0.75)),
    ),
)
len(geometry.primary_atoms), len(geometry.extension_atoms), len(geometry.bonds)

## Default colors, sizes, and finishes

ASE-backed element colors and covalent radii are used automatically. `StyleConfig()` selects the ball-and-stick preset: atom radii are scaled globally by `0.4`, ordinary bonds inherit the two endpoint colors and use radius `0.08` Å, atoms use Phong `0.30`, and bonds use Phong `0.0`. `atom_size_scale` and `bond_size_scale` can independently adjust all resolved atom and bond radii. Use `StyleConfig(preset_style='space_filling')` for full-size atoms without rendered bonds. Only the depth shading needs configuration here.

In [ ]:
styles = StyleConfig(
    depth_shading=DepthShading(
        origin=(0.0, 0.0, 24.0),
        direction=(0.0, 0.0, -1.0),
        decay_length=30.0,
        target=Color(1.0, 1.0, 1.0),
    ),
)
styled = apply_styles(geometry, styles)

## Legacy camera and lighting

The light settings below translate `global_lighting_side.pov`: position `(-4000, -6000, 6000)`, intensity 1.8, a 9×9 soft source spanning 35°, adaptive level 3, global ambient light 0.1, and sphere Phong parameters 0.3/10. The camera remains the side-view camera already reconstructed from the original scene.

In [ ]:
camera = Camera.orthographic(
    direction=(0.0, 100.0, 0.0),
    target=(0.0, 0.0, 21.0),
    up=(0.0, 0.0, 1.0),
    width=21.0,
)
legacy_light = AreaLight(
    location=(-4000.0, -6000.0, 6000.0),
    target=camera.target,
    intensity=1.8,
    angular_diameter=35.0,
    samples=(9, 9),
    adaptive=3,
)
scene = make_scene(
    styled.primitives,
    camera=camera,
    lights=(legacy_light,),
    ambient_light=Color(0.10, 0.10, 0.10),
    background=Background(Color(1.0, 1.0, 1.0, alpha=0.0)),
)

## Render with the legacy INI settings

This uses the original 1024×768 output, quality 5 (required for the area light), gamma 2.0, recursive antialiasing with threshold 0.05, PNG output, and an alpha channel. Once this regression render matches, width and height can be changed independently.

In [ ]:
render_config = RenderConfig(
    width=1024,
    height=768,
    quality=5,
    antialias=True,
    antialias_threshold=0.05,
    sampling_method=2,
    display_gamma=2.0,
    file_gamma=2.0,
    transparent=True,
    display=True,
    executable=povray,
    povray_version='3.7',
)
result = render_scene(
    scene,
    project / 'hematite_legacy_settings.png',
    render_config,
)
print(result.image_path)
Image(filename=result.image_path)

## Export the POV and INI files without rendering

`write_scene` exports only SDL. `write_ini` writes the matching render settings separately, which is useful when opening the files in the POV-Ray GUI. Render the `.ini` file—not the `.pov` alone—so the gamma, antialiasing, transparency, quality, and resolution settings are retained.

In [ ]:
scene_path = write_scene(
    scene,
    project / 'hematite_notebook.pov',
    width=render_config.width,
    height=render_config.height,
    povray_version=render_config.povray_version,
)
ini_path = write_ini(
    scene_path,
    project / 'hematite_notebook.png',
    render_config,
)
scene_path, ini_path